In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

def scale_features(model_data, selected_features):
    """特征标准化"""
    scaler = StandardScaler()
    numeric_features_to_scale = [col for col in selected_features 
                               if col not in ['region_id', 'cate_id', 'cate_level_id', 'brand_id', 'supplier_id']]
    model_data[numeric_features_to_scale] = scaler.fit_transform(model_data[numeric_features_to_scale])
    return model_data, scaler

def train_ensemble_models(X_train, y_train, X_test, y_test):
    """训练随机森林和梯度提升，并计算集成权重"""
    rf_model = RandomForestRegressor(
        n_estimators=300, max_depth=20, min_samples_split=3, 
        min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1
    )
    gb_model = GradientBoostingRegressor(
        n_estimators=300, max_depth=10, learning_rate=0.05, 
        subsample=0.8, min_samples_split=3, min_samples_leaf=1, random_state=42
    )
    
    rf_model.fit(X_train, y_train)
    gb_model.fit(X_train, y_train)
    
    rf_pred = rf_model.predict(X_test)
    gb_pred = gb_model.predict(X_test)
    
    # 智能权重优化
    stack_features = np.column_stack([rf_pred, gb_pred])
    stack_model = LinearRegression().fit(stack_features, y_test)
    
    rf_weight = stack_model.coef_[0] / (stack_model.coef_[0] + stack_model.coef_[1])
    gb_weight = stack_model.coef_[1] / (stack_model.coef_[0] + stack_model.coef_[1])
    
    ensemble_pred = rf_pred * rf_weight + gb_pred * gb_weight
    
    return rf_model, gb_model, ensemble_pred, rf_pred, gb_pred, rf_weight, gb_weight